# 📊 Vibe Evaluation Notebook

**Comprehensive Evaluation of Interface Beauty Models**

This notebook provides tools for evaluating trained Vibe models on interface beauty datasets with detailed metrics and visualizations.


## 🛠️ Setup and Model Loading

In [ ]:
# Setup Vibe environment
!git clone https://github.com/glibas/Vibe.git
%cd Vibe
!pip install -q -r requirements.txt
!pip install -q -e .

from vibe.data.colab_utils import quick_setup
quick_setup()

In [ ]:
# Load trained model
import torch
from vibe.models import BeautyPredictor
from vibe.evaluation import InterfaceEvaluator
from google.colab import drive

# Mount Google Drive to access saved models
drive.mount('/content/drive')

# Specify path to your trained model
model_path = '/content/drive/MyDrive/vibe_training_XXXXXX/best_model.pth'  # Update with your path

# Create model with same configuration as training
model = BeautyPredictor(
    model_type='saliency_guided',
    img_size=224,
    embed_dim=512,
    n_layers=8,
    n_heads=8,
    use_beauty_tokens=True,
    saliency_weight=0.5
)

# Load trained weights
device = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Model loaded from: {model_path}")
    print(f"📊 Trained for {checkpoint.get('epoch', 'unknown')} epochs")
except:
    print("⚠️ Could not load model, using untrained model for demo")

model.to(device)
model.eval()
print(f"🔧 Model ready on: {device}")

## 📊 Dataset Evaluation

In [ ]:
# Load evaluation dataset
from vibe.data.colab_utils import load_webdesign_dataset, setup_webdesign_dataset
from vibe.data import InterfaceDataLoader

# Load dataset
dataset_path = load_webdesign_dataset(
    '/content/drive/MyDrive/datasets/webdesignprototypicality.zip'
)

# Setup dataset paths
data_paths = setup_webdesign_dataset(dataset_path)

# Create test data loader
try:
    # Try to use test split if available
    test_annotations = '/content/drive/MyDrive/vibe_training_XXXXXX/test_annotations.csv'
    test_loader = InterfaceDataLoader.create_val_loader(
        data_path=data_paths['dataset_root'],
        annotations_file=test_annotations,
        batch_size=16
    )
    print(f"📊 Using test set: {len(test_loader.dataset)} samples")
except:
    # Fallback to sample data
    from vibe.data import create_sample_dataset
    create_sample_dataset('/content/eval_data', n_samples=50)
    test_loader = InterfaceDataLoader.create_val_loader(
        data_path='/content/eval_data',
        annotations_file='/content/eval_data/annotations.csv',
        batch_size=16
    )
    print(f"📊 Using sample data: {len(test_loader.dataset)} samples")

In [ ]:
# Run comprehensive evaluation
evaluator = InterfaceEvaluator(model, device=device)

print("🔄 Running evaluation...")
print("=" * 50)

# Evaluate on test set
results = evaluator.evaluate_dataset(
    test_loader,
    save_dir='/content/evaluation_results'
)

print("\n✅ Evaluation completed!")
print("📁 Results saved to: /content/evaluation_results")

## 📈 Results Analysis

In [ ]:
# Display evaluation metrics
import json
import matplotlib.pyplot as plt
import numpy as np

print("📊 EVALUATION RESULTS")
print("=" * 50)

# Overall performance
overall = results['overall']
print(f"🎯 Overall Performance:")
print(f"  RMSE: {overall['rmse']:.4f}")
print(f"  MAE: {overall['mae']:.4f}")
print(f"  Pearson Correlation: {overall['pearson']:.4f}")
print(f"  Spearman Correlation: {overall['spearman']:.4f}")

# Beauty aspects performance
if 'aspects' in results and results['aspects']:
    print(f"\n🎨 Beauty Aspects Performance:")
    for aspect, metrics in results['aspects'].items():
        print(f"  {aspect.capitalize()}:")
        print(f"    MAE: {metrics['mae']:.4f}")
        print(f"    Pearson: {metrics['pearson']:.4f}")

# Design principles performance
if 'principles' in results and results['principles']:
    print(f"\n🏗️ Design Principles Performance:")
    for principle, metrics in results['principles'].items():
        print(f"  {principle.capitalize()}:")
        print(f"    MAE: {metrics['mae']:.4f}")
        print(f"    Pearson: {metrics['pearson']:.4f}")

In [ ]:
# Create visualization plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Vibe Model Evaluation Results', fontsize=16, fontweight='bold')

# Plot 1: Prediction vs Ground Truth
pred = overall['predictions']
gt = overall['ground_truth']

axes[0, 0].scatter(gt, pred, alpha=0.6, s=30, color='blue')
axes[0, 0].plot([0, 1], [0, 1], 'r--', linewidth=2)
axes[0, 0].set_xlabel('Ground Truth Beauty Score')
axes[0, 0].set_ylabel('Predicted Beauty Score')
axes[0, 0].set_title(f'Prediction vs Ground Truth\nPearson: {overall["pearson"]:.3f}')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = pred - gt
axes[0, 1].hist(residuals, bins=20, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Prediction Error')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'Residuals Distribution\nMean: {np.mean(residuals):.3f}')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Beauty Aspects Performance
if 'aspects' in results and results['aspects']:
    aspects = list(results['aspects'].keys())
    mae_scores = [results['aspects'][asp]['mae'] for asp in aspects]
    
    axes[1, 0].bar(aspects, mae_scores, color='skyblue', edgecolor='navy')
    axes[1, 0].set_xlabel('Beauty Aspects')
    axes[1, 0].set_ylabel('Mean Absolute Error')
    axes[1, 0].set_title('Beauty Aspects MAE')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].grid(True, alpha=0.3)
else:
    axes[1, 0].text(0.5, 0.5, 'No aspect data available', 
                   ha='center', va='center', transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('Beauty Aspects (Not Available)')

# Plot 4: Design Principles Performance
if 'principles' in results and results['principles']:
    principles = list(results['principles'].keys())
    corr_scores = [results['principles'][pri]['pearson'] for pri in principles]
    
    axes[1, 1].bar(principles, corr_scores, color='lightcoral', edgecolor='darkred')
    axes[1, 1].set_xlabel('Design Principles')
    axes[1, 1].set_ylabel('Pearson Correlation')
    axes[1, 1].set_title('Design Principles Correlation')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'No principles data available',
                   ha='center', va='center', transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('Design Principles (Not Available)')

plt.tight_layout()
plt.savefig('/content/evaluation_results/comprehensive_results.png', 
           dpi=300, bbox_inches='tight')
plt.show()

print("📊 Visualization saved to: /content/evaluation_results/comprehensive_results.png")

## 🔍 Detailed Sample Analysis

In [ ]:
# Analyze individual samples
print("🔍 DETAILED SAMPLE ANALYSIS")
print("=" * 50)

# Run detailed analysis on sample images
analyses = evaluator.analyze_predictions(
    test_loader, 
    n_samples=5,
    save_dir='/content/sample_analysis'
)

print(f"\n📁 Detailed analysis saved to: /content/sample_analysis")
print(f"📊 Analyzed {len(analyses)} samples")

# Display summary of analyses
for i, analysis in enumerate(analyses):
    print(f"\n🖼️ Sample {i+1}:")
    print(f"  Image: {analysis['image_path']}")
    print(f"  Beauty Score: {analysis['overall_beauty_score']:.3f}")
    if 'ground_truth_beauty' in analysis:
        error = abs(analysis['overall_beauty_score'] - analysis['ground_truth_beauty'])
        print(f"  Ground Truth: {analysis['ground_truth_beauty']:.3f}")
        print(f"  Error: {error:.3f}")

## 📝 Generate Evaluation Report

In [ ]:
# Generate comprehensive evaluation report
report = evaluator.get_summary_report()

print("📝 COMPREHENSIVE EVALUATION REPORT")
print("=" * 60)
print(report)

# Save report to file
with open('/content/evaluation_results/evaluation_report.txt', 'w') as f:
    f.write(report)

print(f"\n📄 Report saved to: /content/evaluation_results/evaluation_report.txt")

## 💾 Save Results to Google Drive

In [ ]:
# Save evaluation results to Google Drive
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_save_dir = f'/content/drive/MyDrive/vibe_evaluation_{timestamp}'

# Copy evaluation results to Drive
if os.path.exists('/content/evaluation_results'):
    shutil.copytree('/content/evaluation_results', drive_save_dir)
    print(f"✅ Evaluation results saved to: {drive_save_dir}")
    
    # List saved files
    print("\n📁 Saved files:")
    for root, dirs, files in os.walk(drive_save_dir):
        for file in files:
            rel_path = os.path.relpath(os.path.join(root, file), drive_save_dir)
            print(f"  {rel_path}")
else:
    print("⚠️ No evaluation results to save")

print("\n🎯 Evaluation completed successfully!")
print("\nKey metrics:")
print(f"  📊 RMSE: {overall['rmse']:.4f}")
print(f"  📈 Pearson: {overall['pearson']:.4f}")
print(f"  📉 MAE: {overall['mae']:.4f}")